<a href="https://colab.research.google.com/github/Kunal-3004/flaskProject/blob/master/Copy_of_notebookc54c097430.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

playground_series_s4e10_path = kagglehub.competition_download('playground-series-s4e10')

print('Data source import complete.')



100%|██████████| 1.45M/1.45M [00:01<00:00, 1.45MB/s]

Extracting files...
Data source import complete.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df=pd.read_csv(playground_series_s4e10_path + '/train.csv')
df.head()

,id,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_status
0,0,37,35000,RENT,0.0,EDUCATION,B,6000,11.49,0.17,N,14,0
1,1,22,56000,OWN,6.0,MEDICAL,C,4000,13.35,0.07,N,2,0
2,2,29,28800,OWN,8.0,PERSONAL,A,6000,8.90,0.21,N,10,0
3,3,30,70000,RENT,14.0,VENTURE,B,12000,11.11,0.17,N,5,0
4,4,22,60000,RENT,2.0,MEDICAL,A,6000,6.92,0.10,N,3,0


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

import gc
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

In [ ]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 10.5 MB/s eta 0:00:00


In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split, cross_val_score,StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression, Ridge
from xgboost import XGBClassifier, XGBRFClassifier, DMatrix
from catboost import CatBoostClassifier, Pool
from lightgbm import LGBMClassifier, early_stopping
from sklearn.ensemble import HistGradientBoostingClassifier


In [ ]:
train_df=df.copy()

In [ ]:
test_df=pd.read_csv(playground_series_s4e10_path + '/test.csv')

In [ ]:
train_df = train_df.reset_index(drop=True)

In [ ]:
train_df.duplicated().sum()

0

In [ ]:
train_df.drop_duplicates(inplace=True)

In [ ]:
train_df.isna().sum()

,0
id,0
person_age,0
person_income,0
person_home_ownership,0
person_emp_length,0
loan_intent,0
loan_grade,0
loan_amnt,0
loan_int_rate,0
loan_percent_income,0


In [ ]:
train_df['loan_int_rate'] = train_df['loan_int_rate'].fillna(train_df['loan_int_rate'].mean())
train_df['person_emp_length'] = train_df['person_emp_length'].fillna(train_df['person_emp_length'].mean())
target = 'loan_status'


In [ ]:
features = train_df.drop(target, axis=1).columns.tolist()
categorical_features = train_df.select_dtypes(include='object').columns.tolist()
numerical_features = list(set(features) - set(categorical_features))
train_df.describe().T

,count,mean,std,min,25%,50%,75%,max
id,58645.0,29322.000000,16929.497605,0.00,14661.00,29322.00,43983.00,58644.00
person_age,58645.0,27.550857,6.033216,20.00,23.00,26.00,30.00,123.00
person_income,58645.0,64046.172871,37931.106979,4200.00,42000.00,58000.00,75600.00,1900000.00
person_emp_length,58645.0,4.701015,3.959784,0.00,2.00,4.00,7.00,123.00
loan_amnt,58645.0,9217.556518,5563.807384,500.00,5000.00,8000.00,12000.00,35000.00
loan_int_rate,58645.0,10.677874,3.034697,5.42,7.88,10.75,12.99,23.22
loan_percent_income,58645.0,0.159238,0.091692,0.00,0.09,0.14,0.21,0.83
cb_person_cred_hist_length,58645.0,5.813556,4.029196,2.00,3.00,4.00,8.00,30.00
loan_status,58645.0,0.142382,0.349445,0.00,0.00,0.00,0.00,1.00


In [ ]:
train_df[categorical_features].describe(include='O').T
train_new=train_df.copy()

In [ ]:
for col in categorical_features:
    train_new[col], _ = train_new[col].factorize()
cor_mat = train_new.corr()
mask = np.triu(cor_mat)

In [ ]:
def model_trainer(model,X,y,test,n_splits=5,random_state=42):
  skfold=StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=random_state)

  roc_aucs=[]
  test_pred=np.zeros(len(test))
  oof_train_preds=np.zeros(len(y))

  model_name=model[-1].__class__.__name__ if isinstance(model, Pipeline) else model.__class__.__name__

  print("="*72)
  print(f"Training {model_name}")
  print("="*72,sep='\n')

  for fold,(train_idx,test_idx) in enumerate(skfold.split(X,y)):
    X_train,y_train=X.iloc[train_idx,:],y[train_idx]
    X_test,y_test=X.iloc[test_idx,:],y[test_idx]

    model_clone=clone(model)
    model_clone.fit(X_train,y_train)
    try:
      y_pred_proba=model_clone.predict_proba(X_test)[:,1]
      test_pred+=model_clone.predict_proba(test)[:,1]
    except:
      y_pred_proba=model_clone.predict(X_test)
      test_pred+=model_clone.predict(test)
    oof_train_preds[test_idx]=y_pred_proba
    roc_auc=roc_auc_score(y_test,y_pred_proba)
    roc_aucs.append(roc_auc)
    print(f"Fold {fold+1} --> ROC_AUC Score: {roc_auc:.4f}")

    del model_clone,X_train,y_train,X_test,y_test
    gc.collect()

  print(f"\nAverage Fold ROC_AUC Score: {np.mean(roc_aucs):.4f} \xb1 {np.std(roc_aucs):.4f}\n")
  return test_pred/skfold.get_n_splits(), oof_train_preds

In [ ]:
def convert_to_string(df):
    df_cat=df.copy()
    df_cat=df_cat.fillna(0)
    for col in df_cat.columns:
        df_cat[col]=df_cat[col].astype(str)
    return df_cat

In [ ]:
skfold=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
X=train_df.drop(target,axis=1)
y=train_df[target].ravel()

In [ ]:
train_preds={}
test_preds={}

In [ ]:
X_xgb=X.copy()
X_xgb[categorical_features]=X_xgb[categorical_features].astype('category')

test_xgb=test_df.copy()
test_xgb[categorical_features]=test_xgb[categorical_features].astype('category')

In [ ]:
oof_preds=[]
oof_aucs=[]
oof_train_preds=np.zeros(len(y))

In [ ]:
xgb_params={
    'eval_metric': 'auc',
    'n_estimators': 5000,
    'learning_rate': 0.10999995057648126,
    'max_depth': 8,
    'subsample':0.999999999999,
    'colsample_bytree': 0.49681888251576845,
    'reg_alpha': 8.919028743833996e-06,
    'reg_lambda': 9.999999999999998,
    'gamma': 0.09787383664032377,
    'min_child_weight': 1.0,
    'max_bin': 262143,
    'enable_categorical': True,
    'early_stopping_rounds': 100,
}

In [ ]:
for fold, (train_idx, test_idx) in enumerate(skfold.split(X_xgb, y)):
    X_train, y_train = X_xgb.iloc[train_idx], y[train_idx]
    X_test, y_test = X_xgb.iloc[test_idx], y[test_idx]

    xgb_clf = XGBClassifier(**xgb_params)
    xgb_clf = xgb_clf.fit(X_train, y_train,
                          eval_set=[(X_test, y_test)],
                          verbose=0)

    booster = xgb_clf.get_booster()
    oof_train_preds[test_idx] = booster.predict(DMatrix(X_test, enable_categorical=True),
                                               iteration_range=(0, xgb_clf.best_iteration + 1))
    test_pred = booster.predict(DMatrix(test_xgb, enable_categorical=True),
                                iteration_range=(0, xgb_clf.best_iteration + 1))
    auc = xgb_clf.best_score
    oof_aucs.append(auc)
    oof_preds.append(test_pred)
    print(f"Fold {fold+1} --> ROC-AUC Score: {auc:.6f}")

    del X_train, y_train, X_test, y_test, xgb_clf
    gc.collect()

auc_mean = np.mean(oof_aucs)
auc_std = np.std(oof_aucs)
print(f"\nAverage Fold ROC-AUC Score: {auc_mean:.6f} \xB1 {auc_std:.6f}\n")

train_preds['xgb'] = oof_train_preds
test_pred_xgb = np.mean(oof_preds, axis=0)
test_preds['xgb'] = test_pred_xgb

Fold 1 --> ROC-AUC Score: 0.952218
Fold 2 --> ROC-AUC Score: 0.964262
Fold 3 --> ROC-AUC Score: 0.956598
Fold 4 --> ROC-AUC Score: 0.960928
Fold 5 --> ROC-AUC Score: 0.960037

Average Fold ROC-AUC Score: 0.958808 ± 0.004100



In [ ]:
oof_preds = []
oof_aucs = []
oof_train_preds = np.zeros(len(y))


In [ ]:
cat_params={
    'iterations': 10000,
    'depth': 10,
    'eta': 0.028901888228959255,
    'reg_lambda': 41.0642500499563,
    'colsample_bylevel': 0.6,
    'subsample': 0.8,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_state': 2024,
    'min_data_in_leaf': 51,
    'early_stopping_rounds': 150,
    'verbose':0,
    'cat_features': features,
    'random_strength': 1.5,
    'bootstrap_type': 'Bernoulli',
    'allow_writing_files': False,
}


In [ ]:
X_cat=convert_to_string(X)
test_cat=convert_to_string(test_df)

test_pool=Pool(test_cat,cat_features=features)

for fold, (train_idx, test_idx) in enumerate(skfold.split(X_cat, y)):
  X_train, y_train=X_cat.iloc[train_idx],y[train_idx]
  X_test, y_test=X_cat.iloc[test_idx],y[test_idx]

  X_train_pool=Pool(X_train,y_train,cat_features=features)
  X_test_pool=Pool(X_test,y_test,cat_features=features)

  cat_clf = CatBoostClassifier(**cat_params)
  cat_clf = cat_clf.fit(X=X_train_pool,
                          eval_set=X_test_pool,
                          verbose=0,
                          early_stopping_rounds=200)
  oof_train_preds[test_idx] = cat_clf.predict_proba(Pool(X_test, cat_features=features))[:, 1]
  test_pred = cat_clf.predict_proba(test_pool)[:, 1]

  oof_preds.append(test_pred)
  auc = cat_clf.best_score_['validation']['AUC']
  oof_aucs.append(auc)
  print(f"\nFold {fold+1}--> ROC-AUC Score: {auc:.6f}\n")

  del X_train, y_train, X_test, y_test
  del X_train_pool, X_test_pool
  del cat_clf
  gc.collect()

auc_mean = np.mean(oof_aucs)
auc_std = np.std(oof_aucs)
print(f"\nAverage Fold ROC-AUC Score: {auc_mean:.6f} \xB1 {auc_std:.6f}\n")

train_preds['cat'] = oof_train_preds
test_pred_cat = np.mean(oof_preds, axis=0)
test_preds['cat'] = test_pred_cat



Fold 1--> ROC-AUC Score: 0.958616


Fold 2--> ROC-AUC Score: 0.968002


Fold 3--> ROC-AUC Score: 0.964457


Fold 4--> ROC-AUC Score: 0.967310


Fold 5--> ROC-AUC Score: 0.964235


Average Fold ROC-AUC Score: 0.964524 ± 0.003312



In [ ]:
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_oe = X.copy()
X_oe[categorical_features] = oe.fit_transform(X_oe[categorical_features])
test_oe = test_df.copy()
test_oe[categorical_features] = oe.transform(test_oe[categorical_features])

hgb_clf = HistGradientBoostingClassifier()
test_preds['hgb'], train_preds['hgb'] = model_trainer(hgb_clf, X_oe, y, test_oe, random_state=101)
xgbrf_clf = XGBRFClassifier()

test_preds['rf'], train_preds['rf'] = model_trainer(xgbrf_clf, X_oe, y, test_oe, random_state=101)
test_preds_df = pd.DataFrame(test_preds)
train_preds_df = pd.DataFrame(train_preds)

Training HistGradientBoostingClassifier
Fold 1 --> ROC_AUC Score: 0.9565
Fold 2 --> ROC_AUC Score: 0.9535
Fold 3 --> ROC_AUC Score: 0.9472
Fold 4 --> ROC_AUC Score: 0.9476
Fold 5 --> ROC_AUC Score: 0.9552

Average Fold ROC_AUC Score: 0.9520 ± 0.0039

Training XGBRFClassifier
Fold 1 --> ROC_AUC Score: 0.9273
Fold 2 --> ROC_AUC Score: 0.9247
Fold 3 --> ROC_AUC Score: 0.9160
Fold 4 --> ROC_AUC Score: 0.9196
Fold 5 --> ROC_AUC Score: 0.9260

Average Fold ROC_AUC Score: 0.9227 ± 0.0042



In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
ridge = HistGradientBoostingClassifier(random_state=42)


ridge_pred_stack, _ = model_trainer(ridge, train_preds_df, y, test_preds_df,random_state=101)

Training HistGradientBoostingClassifier
Fold 1 --> ROC_AUC Score: 0.9672
Fold 2 --> ROC_AUC Score: 0.9633
Fold 3 --> ROC_AUC Score: 0.9607
Fold 4 --> ROC_AUC Score: 0.9640
Fold 5 --> ROC_AUC Score: 0.9669

Average Fold ROC_AUC Score: 0.9644 ± 0.0024



In [ ]:
sub = pd.read_csv(playground_series_s4e10_path + '/sample_submission.csv')
sub[target] = ridge_pred_stack
sub.to_csv('submission.csv', index=False)